In [0]:
%run ./_utils

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

date_str = datetime.now().strftime("%Y-%m-%d")
s3_base_path = f"s3://openalex-snapshots/full/{date_str}"

ENTITIES = [
    {
        "name": "continents",
        "source_table": "openalex.common.continents_api",
        "snapshot_table": "openalex.common.openalex_continents_snapshot",
        "array_columns": ["display_name_alternatives", "countries"],
    },
    {
        "name": "countries",
        "source_table": "openalex.common.countries_api",
        "snapshot_table": "openalex.common.openalex_countries_snapshot",
        "array_columns": ["display_name_alternatives"],
    },
    {
        "name": "institution-types",
        "source_table": "openalex.common.institution_types_api",
        "snapshot_table": "openalex.common.openalex_institution_types_snapshot",
        "array_columns": [],
    },
    {
        "name": "languages",
        "source_table": "openalex.common.languages_api",
        "snapshot_table": "openalex.common.openalex_languages_snapshot",
        "array_columns": [],
    },
    {
        "name": "licenses",
        "source_table": "openalex.common.licenses_api",
        "snapshot_table": "openalex.common.openalex_licenses_snapshot",
        "array_columns": [],
    },
    {
        "name": "sdgs",
        "source_table": "openalex.common.sdgs_api",
        "snapshot_table": "openalex.common.openalex_sdgs_snapshot",
        "array_columns": [],
    },
    {
        "name": "source-types",
        "source_table": "openalex.common.source_types_api",
        "snapshot_table": "openalex.common.openalex_source_types_snapshot",
        "array_columns": [],
    },
    {
        "name": "work-types",
        "source_table": "openalex.common.work_types_api",
        "snapshot_table": "openalex.common.openalex_work_types_snapshot",
        "array_columns": [],
    },
]

### Export each entity to JSONL + Parquet on S3
Writes both formats under `s3://openalex-snapshots/full/{date}/{format}/{entity}/`.

In [0]:
date_str = get_snapshot_date()
for entity in ENTITIES:
    name = entity["name"]
    snapshot_table = entity["snapshot_table"]

    # Read source, apply id_transform if defined, coalesce null arrays
    src = spark.read.table(entity["source_table"])
    if "original_id" in src.columns:
        src = src.drop("original_id")
    id_transform = entity.get("id_transform")
    if id_transform:
        src = id_transform(src)
    from pyspark.sql import functions as F
    for col_name in entity.get("array_columns", []):
        src = src.withColumn(col_name, F.coalesce(F.col(col_name), F.array()))

    # Write the canonical snapshot table
    src.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(snapshot_table)

    # Export both formats via shared utility
    df = spark.read.table(snapshot_table)
    export_partitioned_all_formats(spark, dbutils, df, date_str, name, salt=False)

print("All entities exported.")